# 12 — Comparativa de Modelos
Dataset combinado red + físicos (706.319 registros, features seleccionadas)
Modelos: Random Forest, Logistic Regression, GradientBoosting, XGBoost, LightGBM, CatBoost, MLP
Split estratificado 80/20 con shuffle — mismas métricas para todos

In [0]:
# Instalar librerías que pueden no estar en el entorno serverless
import subprocess
subprocess.run(["pip", "install", "xgboost", "lightgbm", "catboost", "-q"])

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, roc_curve
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import time

DELTA_COMBINED = "/Volumes/workspace/default/network_data/features_combined/"

df = spark.read.format("delta").load(DELTA_COMBINED)
print(f"Total registros : {df.count():,}")
print(f"Columnas        : {len(df.columns)}")

## 1 — Seleccionar features y preparar datos

In [0]:
exclude = [
    "window_id", "window_start", "window_end",
    "session_id", "label",
    "write_read_ratio", "min_payload_bytes",
    "max_payload_bytes", "write_read_ratio_safe",
    "lit401_fit201_ratio", "fit101_fit201_ratio", "lit301_high",
]

feature_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in
       ("double", "long", "int", "integer", "float")
]

print(f"Features seleccionadas : {len(feature_cols)}")

pdf = df.select(feature_cols + ["label"]).toPandas()
pdf[feature_cols] = pdf[feature_cols].fillna(0).astype(float)
pdf["label"]      = pdf["label"].astype(int)

X = pdf[feature_cols].values
y = pdf["label"].values

# Escalar para Logistic Regression y MLP
# Lo hacemos sobre todo X — en cada fold solo ajustaremos sobre train
from sklearn.preprocessing import StandardScaler
scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(X)

# Peso de clase global — se recalcula en cada fold sobre su train
n_normal      = (y == 0).sum()
n_ataque      = (y == 1).sum()
weight_ataque = round(n_normal / n_ataque, 2)
scale_pos     = weight_ataque

print(f"Total registros : {len(X):,}")
print(f"Normal (0)      : {(y==0).sum():,}  ({(y==0).mean()*100:.2f}%)")
print(f"Ataque (1)      : {(y==1).sum():,}  ({(y==1).mean()*100:.2f}%)")
print(f"Peso ataque     : {weight_ataque}")

## 2 — Definir modelos

In [0]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

modelos = {
    "Random Forest": {
        "model": RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_leaf=5,
            class_weight={0: 1.0, 1: weight_ataque},
            n_jobs=-1, random_state=42
        ),
        "scaled": False
    },
    "Logistic Regression": {
        "model": LogisticRegression(
            class_weight="balanced", max_iter=1000,
            random_state=42, n_jobs=-1
        ),
        "scaled": True
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(
            n_estimators=100, max_depth=5,
            learning_rate=0.1, subsample=0.8,
            random_state=42
        ),
        "scaled": False,
        "slow": True  # aviso de lentitud
    },
    "XGBoost": {
        "model": XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos,
            eval_metric="logloss", random_state=42,
            n_jobs=-1, verbosity=0
        ),
        "scaled": False
    },
    "LightGBM": {
        "model": LGBMClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            is_unbalance=True, random_state=42,
            n_jobs=-1, verbose=-1
        ),
        "scaled": False
    },
    "CatBoost": {
        "model": CatBoostClassifier(
            iterations=100, depth=6, learning_rate=0.1,
            auto_class_weights="Balanced",
            random_seed=42, verbose=0
        ),
        "scaled": False
    },
    "MLP": {
        "model": MLPClassifier(
            hidden_layer_sizes=(128, 64, 32), activation="relu",
            max_iter=200, early_stopping=True,
            validation_fraction=0.1, random_state=42
        ),
        "scaled": True
    },
}

print(f"Modelos a evaluar : {len(modelos)}")
for nombre, cfg in modelos.items():
    aviso = " ⚠ LENTO" if cfg.get("slow") else ""
    print(f"  - {nombre}{aviso}")

## 3 — Stratified K-Fold Cross Validation (K=5) para todos los modelos

In [0]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve
import time

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Resultados: para cada modelo guardamos métricas de cada fold
resultados_cv = {}
curvas_roc    = {}

print("Iniciando cross validation K=5 para todos los modelos...")
print("=" * 70)

for nombre, config in modelos.items():
    print(f"\n>>> {nombre} {'[LENTO]' if config.get('slow') else ''}")

    # Datos escalados o no según el modelo
    X_uso = X_scaled if config["scaled"] else X

    metricas_folds = []
    all_y_test  = []
    all_y_proba = []
    all_y_pred  = []

    t0 = time.time()

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_uso, y), start=1):

        X_train_f, X_test_f = X_uso[train_idx], X_uso[test_idx]
        y_train_f, y_test_f = y[train_idx],     y[test_idx]

        # Recalcular peso sobre el train de este fold
        n_n_f = (y_train_f == 0).sum()
        n_a_f = (y_train_f == 1).sum()
        w_a_f = round(n_n_f / n_a_f, 2)

        # Actualizar parámetro de peso según el modelo
        modelo = config["model"]
        if hasattr(modelo, "class_weight") and isinstance(
                getattr(modelo, "class_weight", None), dict):
            modelo.set_params(class_weight={0: 1.0, 1: w_a_f})
        if hasattr(modelo, "scale_pos_weight"):
            modelo.set_params(scale_pos_weight=w_a_f)

        modelo.fit(X_train_f, y_train_f)

        y_pred_f  = modelo.predict(X_test_f)
        y_proba_f = modelo.predict_proba(X_test_f)[:, 1]

        cm_f = confusion_matrix(y_test_f, y_pred_f)
        tn_f, fp_f, fn_f, tp_f = cm_f.ravel()

        metricas_folds.append({
            "fold":           fold,
            "auc_roc":        round(roc_auc_score(y_test_f, y_proba_f), 4),
            "auc_pr":         round(average_precision_score(y_test_f, y_proba_f), 4),
            "f1":             round(f1_score(y_test_f, y_pred_f, pos_label=1), 4),
            "deteccion_%":    round(tp_f / (tp_f + fn_f) * 100, 2),
            "falsa_alarma_%": round(fp_f / (fp_f + tn_f) * 100, 2),
            "fn":             fn_f,
        })

        all_y_test.extend(y_test_f)
        all_y_proba.extend(y_proba_f)
        all_y_pred.extend(y_pred_f)

        print(f"  Fold {fold} | AUC-ROC={metricas_folds[-1]['auc_roc']} | "
              f"Deteccion={metricas_folds[-1]['deteccion_%']}% | "
              f"FA={metricas_folds[-1]['falsa_alarma_%']}%")

    t_total = round(time.time() - t0, 1)

    df_folds = pd.DataFrame(metricas_folds)
    all_y_test  = np.array(all_y_test)
    all_y_proba = np.array(all_y_proba)
    all_y_pred  = np.array(all_y_pred)

    # Curva ROC global del modelo (todos los folds)
    fpr, tpr, _ = roc_curve(all_y_test, all_y_proba)
    auc_global  = roc_auc_score(all_y_test, all_y_proba)
    curvas_roc[nombre] = (fpr, tpr, auc_global)

    cm_g = confusion_matrix(all_y_test, all_y_pred)
    tn_g, fp_g, fn_g, tp_g = cm_g.ravel()

    resultados_cv[nombre] = {
        "folds":          df_folds,
        "auc_roc_media":  round(df_folds["auc_roc"].mean(), 4),
        "auc_roc_std":    round(df_folds["auc_roc"].std(), 4),
        "auc_pr_media":   round(df_folds["auc_pr"].mean(), 4),
        "auc_pr_std":     round(df_folds["auc_pr"].std(), 4),
        "f1_media":       round(df_folds["f1"].mean(), 4),
        "deteccion_media":round(df_folds["deteccion_%"].mean(), 2),
        "fa_media":       round(df_folds["falsa_alarma_%"].mean(), 2),
        "fn_total":       fn_g,
        "auc_roc_global": round(auc_global, 4),
        "tiempo_s":       t_total,
        "all_y_test":     all_y_test,
        "all_y_proba":    all_y_proba,
        "all_y_pred":     all_y_pred,
        "modelo_obj":     modelo,
    }

    print(f"  MEDIA | AUC-ROC={resultados_cv[nombre]['auc_roc_media']} "
          f"±{resultados_cv[nombre]['auc_roc_std']} | "
          f"Deteccion={resultados_cv[nombre]['deteccion_media']}% | "
          f"Tiempo={t_total}s")

print("\n" + "=" * 70)
print("Cross validation completado para todos los modelos")

In [0]:
# Decision Tree — Entrenamiento y visualización
#Árbol de decisión sobre el dataset combinado red + físicos.
#Útil para interpretar las reglas de decisión del modelo.

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, roc_curve
)
import matplotlib.pyplot as plt

DELTA_COMBINED = "/Volumes/workspace/default/network_data/features_combined/"

df = spark.read.format("delta").load(DELTA_COMBINED)
print(f"Total registros : {df.count():,}")


exclude = [
    "window_id", "window_start", "window_end",
    "session_id", "label",
    "write_read_ratio", "min_payload_bytes",
    "max_payload_bytes", "write_read_ratio_safe",
    "lit401_fit201_ratio", "fit101_fit201_ratio", "lit301_high",
]

feature_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in
       ("double", "long", "int", "integer", "float")
]

pdf = df.select(feature_cols + ["label"]).toPandas()
pdf[feature_cols] = pdf[feature_cols].fillna(0).astype(float)
pdf["label"]      = pdf["label"].astype(int)

X = pdf[feature_cols].values
y = pdf["label"].values

n_normal      = (y == 0).sum()
n_ataque      = (y == 1).sum()
weight_ataque = round(n_normal / n_ataque, 2)

print(f"Features  : {len(feature_cols)}")
print(f"Registros : {len(X):,}")
print(f"Peso ataque: {weight_ataque}")

## 1 — K-Fold CV K=5

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metricas_folds = []
all_y_test  = []
all_y_proba = []
all_y_pred  = []

print("Decision Tree — K-Fold CV K=5")
print("=" * 55)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train_f, X_test_f = X[train_idx], X[test_idx]
    y_train_f, y_test_f = y[train_idx], y[test_idx]

    n_n_f = (y_train_f == 0).sum()
    n_a_f = (y_train_f == 1).sum()
    w_a_f = round(n_n_f / n_a_f, 2)

    dt = DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=5,
        class_weight={0: 1.0, 1: w_a_f},
        random_state=42
    )
    dt.fit(X_train_f, y_train_f)

    y_pred_f  = dt.predict(X_test_f)
    y_proba_f = dt.predict_proba(X_test_f)[:, 1]

    cm_f = confusion_matrix(y_test_f, y_pred_f)
    tn_f, fp_f, fn_f, tp_f = cm_f.ravel()

    metricas_folds.append({
        "fold":           fold,
        "auc_roc":        round(roc_auc_score(y_test_f, y_proba_f), 4),
        "auc_pr":         round(average_precision_score(y_test_f, y_proba_f), 4),
        "f1":             round(f1_score(y_test_f, y_pred_f, pos_label=1), 4),
        "deteccion_%":    round(tp_f / (tp_f + fn_f) * 100, 2),
        "falsa_alarma_%": round(fp_f / (fp_f + tn_f) * 100, 2),
        "fn":             fn_f,
    })

    all_y_test.extend(y_test_f)
    all_y_proba.extend(y_proba_f)
    all_y_pred.extend(y_pred_f)

    print(f"  Fold {fold} | AUC-ROC={metricas_folds[-1]['auc_roc']} | "
          f"Deteccion={metricas_folds[-1]['deteccion_%']}% | "
          f"FA={metricas_folds[-1]['falsa_alarma_%']}%")

df_folds = pd.DataFrame(metricas_folds)
all_y_test  = np.array(all_y_test)
all_y_proba = np.array(all_y_proba)
all_y_pred  = np.array(all_y_pred)

print("=" * 55)
print(f"  MEDIA | AUC-ROC={df_folds['auc_roc'].mean():.4f} "
      f"±{df_folds['auc_roc'].std():.4f} | "
      f"Deteccion={df_folds['deteccion_%'].mean():.2f}% | "
      f"FA={df_folds['falsa_alarma_%'].mean():.2f}%")


## 2 — Métricas globales

print(classification_report(
    all_y_test, all_y_pred,
    target_names=["Normal (0)", "Ataque (1)"]
))

cm_g = confusion_matrix(all_y_test, all_y_pred)
tn_g, fp_g, fn_g, tp_g = cm_g.ravel()

print(f"AUC-ROC global  : {roc_auc_score(all_y_test, all_y_proba):.4f}")
print(f"AUC-PR global   : {average_precision_score(all_y_test, all_y_proba):.4f}")
print(f"Deteccion       : {tp_g/(tp_g+fn_g)*100:.2f}%")
print(f"Falsa alarma    : {fp_g/(fp_g+tn_g)*100:.2f}%")
print(f"Falsos Negativos: {fn_g:,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(cm_g, cmap="Blues")
plt.colorbar(im, ax=axes[0])
labels_cm = ["Normal (0)", "Ataque (1)"]
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(labels_cm)
axes[0].set_yticklabels(labels_cm)
axes[0].set_xlabel("Prediccion"); axes[0].set_ylabel("Real")
axes[0].set_title("Matriz de Confusion — Decision Tree",
                  fontsize=12, fontweight="bold")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm_g[i,j]:,}", ha="center", va="center",
                     color="white" if cm_g[i,j] > cm_g.max()/2 else "black",
                     fontsize=14, fontweight="bold")

fpr, tpr, _ = roc_curve(all_y_test, all_y_proba)
axes[1].plot(fpr, tpr, color="#4C8BF5", lw=2,
             label=f"AUC = {roc_auc_score(all_y_test, all_y_proba):.4f}")
axes[1].plot([0,1],[0,1],"k--",lw=1)
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].set_title("Curva ROC", fontsize=12, fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()




In [0]:
## 3 — Visualización del árbol de decisión

# Reentrenar sobre todos los datos para visualizar el árbol completo
# En K-Fold el último fold solo tiene el 80% — aquí usamos el 100%
dt_full = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=5,
    class_weight={0: 1.0, 1: weight_ataque},
    random_state=42
)
dt_full.fit(X, y)

# Reglas de las primeras 4 capas en texto plano
# Útil para incluir un fragmento en la memoria del TFG
print("Reglas de decision — primeras 4 capas:")
print("=" * 60)
print(export_text(
    dt_full,
    feature_names=feature_cols,
    max_depth=4
))

In [0]:
# Visualización de las primeras 4 capas del árbol
# max_depth=4 para que sea legible — el árbol completo (depth=10)
# tendría miles de nodos y sería ilegible
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt_full,
    feature_names=feature_cols,
    class_names=["Normal", "Ataque"],
    filled=True,
    max_depth=4,
    fontsize=8,
    impurity=False,   # ocultar Gini para simplificar visualmente
    proportion=False,
    ax=ax
)
ax.set_title("Decision Tree — primeras 4 capas de decision",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
print("=" * 55)
print("RESUMEN — DECISION TREE (K-Fold CV K=5)")
print("=" * 55)
print(f"  Features          : {len(feature_cols)}")
print(f"  max_depth         : 10")
print(f"  Peso clase ataque : {weight_ataque}")
print("-" * 55)
print(f"  AUC-ROC  (media)  : {df_folds['auc_roc'].mean():.4f} "
      f"± {df_folds['auc_roc'].std():.4f}")
print(f"  AUC-PR   (media)  : {df_folds['auc_pr'].mean():.4f} "
      f"± {df_folds['auc_pr'].std():.4f}")
print(f"  Deteccion (media) : {df_folds['deteccion_%'].mean():.2f}%")
print(f"  Falsa alarma      : {df_folds['falsa_alarma_%'].mean():.2f}%")
print(f"  FN global         : {fn_g:,}")
print("-" * 55)
print(f"  Top feature       : {importances.iloc[0]['feature']}")
print(f"  Nodos del arbol   : {dt_full.tree_.node_count:,}")
print(f"  Profundidad real  : {dt_full.get_depth()}")
print("=" * 55)

## 4 — Tabla comparativa

In [0]:
filas = []
for nombre, res in resultados_cv.items():
    filas.append({
        "Modelo":          nombre,
        "AUC-ROC":         f"{res['auc_roc_media']} ± {res['auc_roc_std']}",
        "AUC-PR":          f"{res['auc_pr_media']} ± {res['auc_pr_std']}",
        "F1 Ataque":       res["f1_media"],
        "Deteccion %":     res["deteccion_media"],
        "Falsa Alarma %":  res["fa_media"],
        "FN (global)":     res["fn_total"],
        "Tiempo (s)":      res["tiempo_s"],
    })

df_comparativa = pd.DataFrame(filas).sort_values(
    "AUC-ROC", ascending=False)

print("Ranking de modelos (ordenado por AUC-ROC media):")
print(df_comparativa.to_string(index=False))

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colores = ["#4C8BF5", "#E8453C", "#059669", "#F59E0B",
           "#7C3AED", "#DC2626", "#1a1a2e"]

# Curvas ROC globales de todos los modelos
for (nombre, (fpr, tpr, auc_val)), color in zip(curvas_roc.items(), colores):
    axes[0].plot(fpr, tpr, lw=1.8, color=color,
                 label=f"{nombre} ({auc_val:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("Tasa Falsos Positivos")
axes[0].set_ylabel("Tasa Verdaderos Positivos")
axes[0].set_title("Curvas ROC globales — todos los modelos",
                  fontsize=12, fontweight="bold")
axes[0].legend(fontsize=8, loc="lower right")
axes[0].grid(alpha=0.3)

# Barras de detección y falsa alarma
nombres = [f["Modelo"] for f in filas]
detecciones = [f["Deteccion %"] for f in filas]
fa = [f["Falsa Alarma %"] for f in filas]

x = np.arange(len(nombres))
w = 0.35
bars1 = axes[1].bar(x - w/2, detecciones, w,
                    color="#059669", alpha=0.85, label="Deteccion (%)")
bars2 = axes[1].bar(x + w/2, fa, w,
                    color="#E8453C", alpha=0.85, label="Falsa Alarma (%)")

axes[1].set_xticks(x)
axes[1].set_xticklabels(nombres, rotation=20, ha="right", fontsize=9)
axes[1].set_title("Deteccion vs Falsa Alarma — media K=5",
                  fontsize=12, fontweight="bold")
axes[1].set_ylabel("%")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f"{bar.get_height():.1f}%",
                 ha="center", va="bottom", fontsize=7)

plt.suptitle("Comparativa de modelos con K-Fold CV (K=5)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5 — Detalle del mejor modelo

In [0]:
# El mejor modelo por AUC-ROC global (todos los folds combinados)
mejor_nombre = max(
    resultados_cv,
    key=lambda n: resultados_cv[n]["auc_roc_global"]
)
mejor = resultados_cv[mejor_nombre]

print(f"Mejor modelo : {mejor_nombre}")
print(f"AUC-ROC      : {mejor['auc_roc_global']}")
print()

print(classification_report(
    mejor["all_y_test"], mejor["all_y_pred"],
    target_names=["Normal (0)", "Ataque (1)"]
))

cm_best = confusion_matrix(mejor["all_y_test"], mejor["all_y_pred"])
tn_b, fp_b, fn_b, tp_b = cm_best.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_best, cmap="Blues")
plt.colorbar(im, ax=ax)
labels_cm = ["Normal (0)", "Ataque (1)"]
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(labels_cm)
ax.set_yticklabels(labels_cm)
ax.set_xlabel("Prediccion", fontsize=12)
ax.set_ylabel("Real", fontsize=12)
ax.set_title(f"Matriz de Confusion — {mejor_nombre} (global K=5)",
             fontsize=12, fontweight="bold")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_best[i,j]:,}",
                ha="center", va="center",
                color="white" if cm_best[i,j] > cm_best.max()/2 else "black",
                fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
y_proba_best = mejor["all_y_proba"]
y_test_best  = mejor["all_y_test"]

thresholds_range = np.arange(0.1, 0.9, 0.05)
results_thresh = []

for t in thresholds_range:
    y_pred_t = (y_proba_best >= t).astype(int)
    cm_t = confusion_matrix(y_test_best, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    results_thresh.append({
        "threshold":        round(t, 2),
        "recall_ataque":    round(tp_t / (tp_t + fn_t) * 100, 2),
        "precision_ataque": round(tp_t / (tp_t + fp_t) * 100, 2) if (tp_t + fp_t) > 0 else 0,
        "falsa_alarma":     round(fp_t / (fp_t + tn_t) * 100, 2),
        "f1_ataque":        round(2 * tp_t / (2 * tp_t + fp_t + fn_t) * 100, 2) if (2*tp_t + fp_t + fn_t) > 0 else 0
    })

df_thresh = pd.DataFrame(results_thresh)
print(f"Ajuste de umbral — {mejor_nombre}:")
print(df_thresh.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_thresh["threshold"], df_thresh["recall_ataque"],
        color="#E8453C", lw=2, label="Recall Ataque (%)")
ax.plot(df_thresh["threshold"], df_thresh["precision_ataque"],
        color="#4C8BF5", lw=2, label="Precision Ataque (%)")
ax.plot(df_thresh["threshold"], df_thresh["falsa_alarma"],
        color="gray", lw=1.5, linestyle="--", label="Falsa Alarma (%)")
ax.plot(df_thresh["threshold"], df_thresh["f1_ataque"],
        color="#F59E0B", lw=2, label="F1 Ataque (%)")
ax.axvline(0.5, color="black", linestyle=":", lw=1, label="Umbral default (0.5)")
ax.set_xlabel("Umbral de clasificacion")
ax.set_ylabel("%")
ax.set_title(f"Metricas vs Umbral — {mejor_nombre}",
             fontsize=12, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_thresh = df_thresh.loc[df_thresh["f1_ataque"].idxmax()]
print(f"\nUmbral optimo (F1)   : {best_thresh['threshold']}")
print(f"  Recall Ataque      : {best_thresh['recall_ataque']}%")
print(f"  Precision Ataque   : {best_thresh['precision_ataque']}%")
print(f"  Falsa Alarma       : {best_thresh['falsa_alarma']}%")
print(f"  F1 Ataque          : {best_thresh['f1_ataque']}%")

## 6 — Resumen final

In [0]:
print("=" * 70)
print("RESUMEN — COMPARATIVA K-FOLD CV (K=5)")
print("=" * 70)
print(f"  Dataset     : Red + Fisicos ({len(X):,} registros, {len(feature_cols)} features)")
print(f"  Validacion  : Stratified K-Fold K=5, shuffle=True")
print("-" * 70)
print(f"  {'Modelo':<25} {'AUC-ROC':>12} {'Detec%':>8} {'FA%':>6} {'FN':>6} {'t(s)':>7}")
print("-" * 70)
for _, row in df_comparativa.iterrows():
    print(f"  {row['Modelo']:<25} {row['AUC-ROC']:>12} "
          f"{row['Deteccion %']:>7}% {row['Falsa Alarma %']:>5}% "
          f"{row['FN (global)']:>6} {row['Tiempo (s)']:>7}")
print("-" * 70)
print(f"  Mejor modelo (AUC-ROC) : {mejor_nombre}")
print(f"  AUC-ROC global         : {mejor['auc_roc_global']}")
print(f"  Umbral optimo (F1)     : {best_thresh['threshold']}")
print(f"  Recall con optimo      : {best_thresh['recall_ataque']}%")
print(f"  Falsa alarma optimo    : {best_thresh['falsa_alarma']}%")
print("=" * 70)